# Pipeline de procesamiento paralelo — equipo ACC (Soporte Técnico ISP)

Notebook equivalente a `spark/src/pipeline.py`, celda por transformación (Paso 6 /
Módulo E de la Guía de Entrega 3, D4.1 de la rúbrica). Las 5 transformaciones
exigidas, aplicadas al objetivo específico del equipo ACC (Tabla 1 de la guía):
**análisis de reincidencia** y **agrupamiento (clustering) de incidencias por
texto y zona**.

1. Filtrado
2. Join entre tablas colocalizadas (incidencias ⋈ clientes por `client_id`)
3. Agregación con ventanas (reincidencia)
4. Transformación de tipos temporales
5. Operación de ML (StringIndexer + TF-IDF + KMeans)

In [1]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession
from transformations import (
    build_clients_dim,
    t1_filter_valid_resolved,
    t2_join_clients,
    t3_recurrence_window,
    t4_recurrence_flag,
    t5_cluster_by_text_and_zone,
)

ZONES = ["QUEVEDO_CENTRO", "QUEVEDO_NORTE", "QUEVEDO_SUR"]
N_CLIENTS_PER_ZONE = 30_000  # debe coincidir con generate_dataset.N_CLIENTS_PER_ZONE

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("acc-soporte-tecnico-notebook")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/26 19:33:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Carga del dataset

Generado por `spark/src/generate_dataset.py` (sintético, semilla determinista
`seed=42`, ver honestidad sobre el alcance documentada en ese script).

In [2]:
df = spark.read.parquet("../data/processed/incidents")
df.printSchema()
df.count()

root
 |-- incident_id: long (nullable = true)
 |-- node_id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- technician_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- resolved: boolean (nullable = true)
 |-- sla_breached: boolean (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- zone: string (nullable = true)



600000

## T1 — Filtrado

Incidencias resueltas y con los campos que el resto del pipeline necesita
(`client_id`, `description`).

In [3]:
r1 = t1_filter_valid_resolved(df)
r1.cache()
print(f"Filas tras T1: {r1.count():,}")
r1.select("incident_id", "zone", "client_id", "incident_type", "description").show(5, truncate=False)

Filas tras T1: 557,701


+-----------+--------------+----------------------------+-------------+-------------------------------------------------+
|incident_id|zone          |client_id                   |incident_type|description                                      |
+-----------+--------------+----------------------------+-------------+-------------------------------------------------+
|5          |QUEVEDO_CENTRO|QUEVEDO_CENTRO-CLIENTE-10892|HARDWARE     |hay un cable de red visiblemente danado          |
|9          |QUEVEDO_CENTRO|QUEVEDO_CENTRO-CLIENTE-22523|DNS          |puedo hacer ping pero el navegador no abre sitios|
|11         |QUEVEDO_CENTRO|QUEVEDO_CENTRO-CLIENTE-20529|CORTE_TOTAL  |las luces del modem estan apagadas               |
|16         |QUEVEDO_CENTRO|QUEVEDO_CENTRO-CLIENTE-03601|CONFIGURACION|cambie de equipo y no logro conectarlo           |
|22         |QUEVEDO_CENTRO|QUEVEDO_CENTRO-CLIENTE-19650|DNS          |no resuelve nombres de dominio desde esta manana |
+-----------+-----------

## T2 — Join entre tablas colocalizadas

Incidencias ⋈ clientes por `client_id` (Tabla 1 de la guía de E3, columna
"Fragmentación + colocalización" del equipo ACC).

In [4]:
clients_dim = build_clients_dim(spark, ZONES, N_CLIENTS_PER_ZONE)
r2 = t2_join_clients(r1, clients_dim)
r2.cache()
r2.select("incident_id", "client_id", "client_name", "zone").show(5, truncate=False)

26/07/26 19:34:12 WARN TaskSetManager: Stage 9 contains a task of very large size (1132 KiB). The maximum recommended task size is 1000 KiB.


+-----------+----------------------------+-------------+--------------+
|incident_id|client_id                   |client_name  |zone          |
+-----------+----------------------------+-------------+--------------+
|731        |QUEVEDO_CENTRO-CLIENTE-06598|Cliente 06598|QUEVEDO_CENTRO|
|1296       |QUEVEDO_CENTRO-CLIENTE-15997|Cliente 15997|QUEVEDO_CENTRO|
|1567       |QUEVEDO_CENTRO-CLIENTE-29402|Cliente 29402|QUEVEDO_CENTRO|
|1939       |QUEVEDO_CENTRO-CLIENTE-28913|Cliente 28913|QUEVEDO_CENTRO|
|3015       |QUEVEDO_CENTRO-CLIENTE-12999|Cliente 12999|QUEVEDO_CENTRO|
+-----------+----------------------------+-------------+--------------+
only showing top 5 rows


## T3 — Agregación con ventanas (reincidencia)

`Window.partitionBy("client_id").orderBy("timestamp")`: orden y timestamp
anterior de cada cliente.

In [5]:
r3 = t3_recurrence_window(r2)
r3.cache()
r3.select("client_id", "timestamp", "incident_seq", "previous_timestamp").orderBy("client_id", "incident_seq").show(10, truncate=False)

+----------------------------+-------------------+------------+-------------------+
|client_id                   |timestamp          |incident_seq|previous_timestamp |
+----------------------------+-------------------+------------+-------------------+
|QUEVEDO_CENTRO-CLIENTE-00001|2026-06-24 10:11:00|1           |NULL               |
|QUEVEDO_CENTRO-CLIENTE-00002|2026-02-27 07:59:00|1           |NULL               |
|QUEVEDO_CENTRO-CLIENTE-00002|2026-02-28 22:57:00|2           |2026-02-27 07:59:00|
|QUEVEDO_CENTRO-CLIENTE-00002|2026-03-14 21:30:00|3           |2026-02-28 22:57:00|
|QUEVEDO_CENTRO-CLIENTE-00002|2026-03-16 08:11:00|4           |2026-03-14 21:30:00|
|QUEVEDO_CENTRO-CLIENTE-00002|2026-06-04 06:08:00|5           |2026-03-16 08:11:00|
|QUEVEDO_CENTRO-CLIENTE-00003|2026-07-04 22:48:00|1           |NULL               |
|QUEVEDO_CENTRO-CLIENTE-00003|2026-07-17 20:50:00|2           |2026-07-04 22:48:00|
|QUEVEDO_CENTRO-CLIENTE-00004|2026-03-28 14:41:00|1           |NULL         

## T4 — Transformación de tipos temporales

Días desde la incidencia anterior del mismo cliente + bandera de reincidencia
en 30 días.

In [6]:
r4 = t4_recurrence_flag(r3)
r4.cache()
recurring = r4.filter(r4.is_recurring_30d)
print(f"Incidencias reincidentes en 30 dias: {recurring.count():,} de {r4.count():,}")
recurring.select("client_id", "zone", "incident_seq", "days_since_previous").show(10, truncate=False)

Incidencias reincidentes en 30 dias: 380,753 de 557,701
+----------------------------+--------------+------------+-------------------+
|client_id                   |zone          |incident_seq|days_since_previous|
+----------------------------+--------------+------------+-------------------+
|QUEVEDO_CENTRO-CLIENTE-00016|QUEVEDO_CENTRO|3           |21.159027777777776 |
|QUEVEDO_CENTRO-CLIENTE-00223|QUEVEDO_CENTRO|2           |10.165277777777778 |
|QUEVEDO_CENTRO-CLIENTE-00223|QUEVEDO_CENTRO|6           |23.644444444444446 |
|QUEVEDO_CENTRO-CLIENTE-00223|QUEVEDO_CENTRO|7           |6.409027777777778  |
|QUEVEDO_CENTRO-CLIENTE-00223|QUEVEDO_CENTRO|8           |5.108333333333333  |
|QUEVEDO_CENTRO-CLIENTE-00627|QUEVEDO_CENTRO|2           |2.4256944444444444 |
|QUEVEDO_CENTRO-CLIENTE-00627|QUEVEDO_CENTRO|3           |0.9854166666666667 |
|QUEVEDO_CENTRO-CLIENTE-00683|QUEVEDO_CENTRO|2           |7.2659722222222225 |
|QUEVEDO_CENTRO-CLIENTE-00683|QUEVEDO_CENTRO|3           |11.83888888888888

## T5 — Operación de ML: clustering por texto y zona

`Tokenizer` + `StopWordsRemover` + `HashingTF` + `IDF` sobre la descripción
libre, `StringIndexer` sobre la zona, agrupados con `KMeans`.

In [7]:
r5 = t5_cluster_by_text_and_zone(r1, k=5)
r5.cache()
r5.groupBy("cluster").count().orderBy("cluster").show()
r5.select("zone", "incident_type", "description", "cluster").show(10, truncate=False)

26/07/26 19:34:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/07/26 19:34:55 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-------+------+
|cluster| count|
+-------+------+
|      0| 13761|
|      1|453625|
|      2| 27815|
|      3| 13968|
|      4| 48532|
+-------+------+

+--------------+-------------+-------------------------------------------------+-------+
|zone          |incident_type|description                                      |cluster|
+--------------+-------------+-------------------------------------------------+-------+
|QUEVEDO_CENTRO|HARDWARE     |hay un cable de red visiblemente danado          |1      |
|QUEVEDO_CENTRO|DNS          |puedo hacer ping pero el navegador no abre sitios|1      |
|QUEVEDO_CENTRO|CORTE_TOTAL  |las luces del modem estan apagadas               |0      |
|QUEVEDO_CENTRO|CONFIGURACION|cambie de equipo y no logro conectarlo           |2      |
|QUEVEDO_CENTRO|DNS          |no resuelve nombres de dominio desde esta manana |1      |
|QUEVEDO_CENTRO|DNS          |puedo hacer ping pero el navegador no abre sitios|1      |
|QUEVEDO_CENTRO|LENTITUD     |los videos se t

## Cierre

In [8]:
spark.stop()